# BP2 Gate 3 — Model / Classifier Benchmark
**Customer360 Navigator Enterprise Suite — Customer Friction Classification**

## Why this notebook exists
BP2 Gate 2's real run (2026-09-22) tagged every real CFPB row with `friction_severity_class` and
wrote the Gold layer (`data/processed/cfpb_friction_severity_gold.parquet`) — 816,717 real rows
trainable across 4 ordinal classes (`LOW_FRICTION` 10,427 / `MEDIUM_FRICTION` 312,526 /
`MEDIUM_HIGH_FRICTION` 490,537 / `HIGH_FRICTION` 3,227), 231,858 excluded
(`EXCLUDED_PENDING`/`EXCLUDED_UNKNOWN`, never dropped from the Gold layer, only flagged
non-trainable). This notebook is the first to actually train and benchmark classifiers on that
target.

## Feature set — a Gate 3 decision, as Gate 1 explicitly deferred it
Real, structured CFPB fields only:
`Product, Sub-product, Issue, Sub-issue, State, Submitted via, common_taxonomy_bucket` (one-hot
encoded) plus `Company` (frequency-encoded — 2,970 distinct real values, too high-cardinality to
one-hot without exploding dimensionality). Every remaining candidate uses this one shared
preprocessed feature matrix — no raw-categorical path (previously CatBoost's own) remains.

**Barred from the feature set, every one for a documented reason:**
- `Company response to consumer`, `Timely response?` — these DEFINE the target (Gate 1's leakage
  rule #1).
- `Date received`, `Date sent to company` — **newly identified this gate**: these two real fields
  let a duration be computed, and CFPB's own real-world timeliness determination is duration-based
  — so a response-time feature would very likely leak the `Timely response?`-derived half of the
  target almost perfectly. Barred for the same reason as the two fields above, not merely "unused."
- `Company public response` — not pre-cleared by Gate 1/2 (~54% null, untested for leakage).
  Deliberately left OUT of this first benchmark rather than guessed safe; a with/without ablation
  is recorded below as an explicit open item, not run here.
- `Complaint ID` — row identifier, no signal.
- `ZIP code` — 19,031 distinct values, high-cardinality with no real aggregation logic defined, and
  excluded out of the same compliance caution Gate 1 recorded (ECOA/Reg B: "wherever any
  demographic-adjacent field could appear").
- `Tags` — excluded regardless of content (low information, only a few distinct real values), but
  this notebook live-checks its actual real values first and flags explicitly if any is
  demographic-adjacent (e.g. an "Older American"/"Servicemember" style value) — Gate 1's
  compliance statement ("No demographic or protected-class field exists in the real CFPB extract")
  is re-verified live here, not assumed to still hold.

## What this notebook does
1. Live-verifies row counts against Gate 2's documented `gate2` config block (drift check, never
   trusts a baked-in count).
2. Live-checks `Tags`' real values for demographic-adjacent content (compliance re-verification).
3. Builds a single stratified train/test split of the 816,717 trainable rows (CFPB provides no
   split of its own — Gate 1's policy already recorded this: fresh random split, stratified by the
   derived target, `random_state=42`).
4. Benchmarks 5 candidate models (logistic_regression, random_forest,
   hist_gradient_boosting, xgboost, lightgbm) under identical CV folds, with
   `class_weight='balanced'` applied wherever each library's real, installed API supports it for
   multiclass (LogisticRegression, RandomForest, LightGBM —
   HistGradientBoostingClassifier and XGBoost's sklearn API do not expose an equivalent for this
   real installed version, a documented library-capability asymmetry, not an oversight). CatBoost
   was removed from the candidate set — see the "Rewritten" note below.
5. Selects the champion by mean CV macro-F1 (severe class imbalance here — the real trainable set
   is ~60% `MEDIUM_HIGH_FRICTION` vs ~0.4% `HIGH_FRICTION`, a ~152:1 ratio — so macro-F1, not
   accuracy, is the only defensible selection metric; Master Plan Section 5.1's "target F1 ≥ 0.80
   where supported" is a goal, not a guaranteed outcome, and this notebook does not assume it will
   be met).
6. Refits the champion once on the full train split, evaluates once on the real held-out test
   split, writes a confusion matrix and top-confused-pair table across the 4 real severity classes.

## Standing rules this notebook follows
- **Execution boundary**: Claude wrote this notebook; it does not run it.
- **Zero-fabrication**: every row count, drift check, and class distribution below is computed
  live against the real Gold parquet, never assumed from a prior gate's output without
  re-verifying.
- **WARP**: reads `Parquet` (not CSV), `class_weight='balanced'` instead of resampling, sparse
  one-hot matrices, `float32` for the one dense conversion `hist_gradient_boosting` real-environment
  quirk requires (same quirk BP1 Gate 3 found in this same installed sklearn version), reused
  `n_jobs`/CV settings from `hardware_benchmark_summary.json`/`resource_limits.yaml` rather than
  hardcoded. Reads the Gold parquet via Polars (not `pandas.read_parquet`/`polars.to_pandas()`) —
  this environment's own real library scan (`hardware_benchmark_summary.json`) does not confirm
  `pyarrow` is installed, and BP1 Gate 3 already flagged that dependency as an avoidable risk for
  exactly this reason; feature columns are extracted via Polars' own `.to_numpy()`/`.to_list()`
  and reassembled into a plain `pandas.DataFrame` (no Arrow conversion path involved).
- **HYPER**: reuses `bp1_config_sync.py` unmodified for the gate3 config block.

## Incident and comprehensive hardening (2026-09-22) — see LESSONS_LEARNED_APPLIED.md Lesson #21
This notebook's first real run ended in a real Windows crash (Kernel-Power Event 41, unclean
shutdown, reported and confirmed by the user as occurring while this notebook was running). An
earlier, narrower fix — capping only `hist_gradient_boosting`'s CV step to `n_jobs=2` — had
already been applied before that run, based on a code-review finding (not a crash) that its
densified float32 matrix (~2.4GB code-review estimate, never profiled for real) could have up to
`n_splits` copies resident at once under threading-backend CV concurrency. That fix was correct
but insufficient in scope: every other candidate was still running its CV step at the full,
uncapped `N_JOBS` (=16, a SYNTHETIC hardware-benchmark recommendation, not a number derived from
this workload), and a genuinely distinct sustained-CPU/thermal risk — independent of any single
candidate's memory footprint — was not addressed at all.

**Comprehensive fix now applied, covering every candidate and both risk categories the user named
separately ("thermal freeze" vs. "ram or cpu freeze"):**
- **Per-candidate CV concurrency caps for all 6 models**, not just one: `hist_gradient_boosting`
  tightened further to `n_jobs=1` (given the confirmed real crash); `random_forest` and `catboost`
  newly capped to `n_jobs=2` each (real per-fold CPU/memory cost — 100 trees × max_depth=20 for
  the former, per-fold target-statistics computation over 2,970-distinct-value `Company` for the
  latter); every other candidate (`logistic_regression`, `xgboost`, `lightgbm`) capped to
  `min(4, N_JOBS)` rather than left uncapped, since `cv_settings['n_splits']==5` makes any value
  above 5 pointless anyway.
- **A pre-candidate (not just post-candidate) adaptive memory check**: live RAM headroom is read
  immediately before each candidate's CV starts (`memory_headroom_gb()`), and if it has already
  fallen below a 4GB threshold, that candidate is forced to `n_jobs=1` regardless of its planned
  cap — never assumes the previous candidate's cleanup (`del X_cv` + `assert_within_ram_ceiling()`)
  fully freed memory.
- **A 20-second precautionary pause between candidates**, addressing sustained-CPU/thermal load as
  its own risk category: this notebook's real dataset (816,717 trainable rows) is ~63x larger than
  any prior real run in this project (BP1 Gate 3: ~13,083 rows), so total high-CPU wall-clock time
  across 6 candidates is far longer than anything previously run here for real. `psutil` exposes
  no portable CPU temperature reading on Windows, so this pause is a documented precaution, not a
  measured thermal response.
- **Every result row now records `cv_n_jobs_used` and `ram_headroom_gb_before_candidate`**, so the
  real run's own console output and CSV are self-documenting evidence of what concurrency and
  headroom applied to each candidate — not just a pass/fail signal.
- **Model hyperparameters were deliberately left untouched** (`n_estimators`, `max_depth`,
  `iterations`, etc.) — this fix is concurrency/pacing/monitoring only, so it does not compromise
  the real benchmark's fidelity.

**Honesty note, unchanged from the original finding:** this remains a plausible, not a proven,
explanation for the real crash — no Windows crash dump (`Get-WinEvent` stop-code lookup) has been
reviewed. The fix is applied regardless, because the underlying uncapped-concurrency and
no-thermal-pacing conditions are real risks independent of whether they are confirmed as this
specific crash's cause.


## Rewritten 2026-09-22 — CatBoost removed from the candidate set (user decision)
Lesson #22 (`LESSONS_LEARNED_APPLIED.md`) root-caused BP2 Gate 3's `catboost` candidate failure as
a genuine upstream incompatibility: this project's installed scikit-learn (1.8.0) added a strict
post-clone identity check that `CatBoostClassifier.get_params()` can never satisfy when built with
a `cat_features` constructor argument, so `cross_validate()` (which calls `clone()` once per fold)
fails for CatBoost on every real run, regardless of what else changes. A working fix (a manual
per-fold CV loop bypassing `clone()` entirely) was built, applied, and verified — but after
weighing it, the user chose the simpler path instead: **remove CatBoost from the candidate set
entirely** rather than carry a special-cased workaround for one of six candidates going forward.

This notebook now benchmarks **5 candidates** (logistic_regression, random_forest,
hist_gradient_boosting, xgboost, lightgbm) — every one goes through the identical shared
one-hot/frequency-encoded preprocessing and the same `cross_validate()` path; there is no longer a
raw-categorical code path in this notebook at all. `CATBOOST_FEATURE_COLS`,
`CATBOOST_CAT_FEATURE_INDICES`, and `USES_RAW_CATEGORICAL` have been removed rather than left as
unused dead code.

**Known real tradeoff, stated plainly rather than left implicit:** `Company` (2,970 distinct real
values) is exactly the kind of high-cardinality categorical feature CatBoost's native
target-statistics encoding is designed for, more so than the frequency-encoding the remaining 5
candidates use. Removing CatBoost means this benchmark no longer tests the one algorithm built
for that specific feature — a deliberate simplicity-over-completeness tradeoff, not an oversight.

**This notebook must be re-run for real** to get the genuine 5-candidate benchmark. Champion
selection may change versus the previously recorded xgboost (catboost was never champion before,
so removing it cannot itself change who wins among the remaining 5 — but this is still a fresh
real run of `cross_validate()` for every candidate, so treat the champion as unconfirmed until this
real run completes). If the champion changes, Gates 4, 5, 6 and the executive rollup all need
re-running afterward too (champion-agnostic by design — no code changes needed there, only
re-execution). The pre-removal notebook (Lesson #22's clone()-fix version) is preserved as
`...g3_model_benchmark.PRE_CATBOOST_FIX_BACKUP.ipynb`, and the version before that fix
(pre-hardening) as `...g3_model_benchmark.PRE_HARDENING_BACKUP.ipynb` — neither was discarded.


In [ ]:
"""
Customer360 Navigator Enterprise Suite - BP2 Gate 3 model / classifier benchmark notebook.
Single consolidated code cell (platform convention). Idempotent - safe to re-run.
"""

import os, sys, json, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
def _find_project_root() -> Path:
    marker = "PROJECT_STRUCTURE_LOCKED.md"
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        if (Path(env_override) / marker).exists():
            return Path(env_override)
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {env_override!r} but {marker} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    cur = start
    for _ in range(8):
        if (cur / marker).exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker in filenames:
            return Path(depth_root)

    raise RuntimeError(
        f"Could not resolve PROJECT_ROOT: no {marker} found by walking up from {start}, nor by "
        "searching up to 3 levels below it. Fix: add a cell at the TOP of this notebook (before "
        "this cell runs) with:\n"
        '    import os; os.environ["C360_PROJECT_ROOT"] = r"C:\\Users\\rnand\\Documents\\'
        'Customer360_Navigator_Enterprise_Suite"\n'
        "then re-run from the top."
    )

PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP performance configuration - FIRST, before any heavy import
# ============================================================
from utils.performance_setup import (  # noqa: E402
    assert_within_ram_ceiling,
    configure_performance,
    load_resource_limits,
    memory_headroom_gb,
)

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)
RESOURCE_LIMITS = load_resource_limits(PROJECT_ROOT)
assert_within_ram_ceiling(RESOURCE_LIMITS)

# ============================================================
# SECTION 3: Heavy imports + flush-forcing print override
# ============================================================
import builtins  # noqa: E402
import functools  # noqa: E402
import time  # noqa: E402
import yaml  # noqa: E402
from datetime import datetime, timezone  # noqa: E402

import joblib  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import polars as pl  # noqa: E402
from scipy import sparse as sp  # noqa: E402
from sklearn.compose import ColumnTransformer  # noqa: E402
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier  # noqa: E402
from sklearn.linear_model import LogisticRegression  # noqa: E402
from sklearn.metrics import classification_report, confusion_matrix  # noqa: E402
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split  # noqa: E402
from sklearn.preprocessing import LabelEncoder, OneHotEncoder  # noqa: E402
from xgboost import XGBClassifier  # noqa: E402
from lightgbm import LGBMClassifier  # noqa: E402

warnings.filterwarnings("ignore")
print = functools.partial(builtins.print, flush=True)

CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp2_customer_friction_classification" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

GOLD_PATH = DATA_PROCESSED / "cfpb_friction_severity_gold.parquet"
BP2_CONFIG_PATH = CONFIGS_DIR / "bp2_customer_friction_classification.yaml"
SEVERITY_CONFIG_PATH = CONFIGS_DIR / "bp2_friction_severity_taxonomy.yaml"

for p in (GOLD_PATH, BP2_CONFIG_PATH, SEVERITY_CONFIG_PATH):
    if not p.exists():
        raise FileNotFoundError(
            f"Required input not found: {p}. Confirm BP2 Gates 1-2 both completed for real."
        )

# ============================================================
# SECTION 4: Load Gate 1/2 policy - never hardcode what a prior gate already recorded
# ============================================================
with open(BP2_CONFIG_PATH, "r", encoding="utf-8") as f:
    bp2_config = yaml.safe_load(f)
with open(SEVERITY_CONFIG_PATH, "r", encoding="utf-8") as f:
    severity_config = yaml.safe_load(f)

TARGET_COL = bp2_config["target_definition"]["primary_target"]
RANDOM_STATE = bp2_config["random_state"]
ORDINAL_CLASSES = list(severity_config["severity_classes"].keys())
EXCLUDED_CLASSES = list(severity_config["excluded_classes"].keys())
# The gate2 block's keys land at the top level of the parsed YAML (marker-delimited blocks are
# still plain top-level YAML keys, not nested under a "gate2" key) - see bp1_config_sync.py.
documented_trainable_rows = bp2_config["trainable_rows_4_ordinal_classes"]
print(f"[OK] Gate 1/2 policy loaded: target='{TARGET_COL}', random_state={RANDOM_STATE}, "
      f"ordinal_classes={ORDINAL_CLASSES}, excluded_classes={EXCLUDED_CLASSES}, "
      f"documented trainable rows (Gate 2)={documented_trainable_rows:,}")

hw_summary_path = CONFIGS_DIR / "hardware_benchmark_summary.json"
with open(hw_summary_path, "r", encoding="utf-8") as f:
    hw_summary = json.load(f)
N_JOBS = hw_summary["recommended_configuration"]["recommended_n_jobs"]
print(f"[OK] Hardware benchmark recommendation loaded: n_jobs={N_JOBS} (never hardcoded)")

cv_settings = RESOURCE_LIMITS["cv"]
print(f"[OK] CV settings loaded: n_splits={cv_settings['n_splits']}, "
      f"random_state={cv_settings['random_state']}")

# ============================================================
# SECTION 5: Load the real Gold layer via Polars (not pandas.read_parquet/polars.to_pandas() -
# this environment's own real library scan does not confirm pyarrow is installed, and BP1 Gate 3
# already flagged that dependency as an avoidable risk for the same reason).
# ============================================================
FEATURE_COLS_CATEGORICAL = ["Product", "Sub-product", "Issue", "Sub-issue", "State",
                            "Submitted via", "common_taxonomy_bucket"]
COMPANY_COL = "Company"

BARRED_COLUMNS = ["Company response to consumer", "Timely response?", "Date received",
                  "Date sent to company", "Company public response", "Complaint ID", "ZIP code", "Tags"]

gold_lazy = pl.scan_parquet(GOLD_PATH)

# --- Live compliance re-check on 'Tags' - excluded from the feature set regardless of content,
# but Gate 1's "No Protected-Class Field in Scope" statement is re-verified live here, not assumed.
tags_dist = gold_lazy.group_by("Tags").agg(pl.len().alias("n")).sort("n", descending=True).collect()
print("\n[COMPLIANCE CHECK] Live 'Tags' distinct real values (excluded from the feature set regardless):")
print(tags_dist)
DEMOGRAPHIC_ADJACENT_KEYWORDS = ("older american", "servicemember", "military", "veteran")
tags_values_lower = [str(v).lower() for v in tags_dist["Tags"].to_list() if v is not None]
demographic_adjacent_tags_found = [v for v in tags_values_lower
                                    if any(kw in v for kw in DEMOGRAPHIC_ADJACENT_KEYWORDS)]
if demographic_adjacent_tags_found:
    print(f"[COMPLIANCE FLAG] 'Tags' contains real demographic-adjacent value(s): "
          f"{demographic_adjacent_tags_found}. This UPDATES BP2 Gate 1's compliance_touchpoint "
          "statement ('No demographic or protected-class field exists') - 'Tags' is excluded from "
          "the feature set here regardless, but this finding must be reviewed against ECOA/"
          "Regulation B before Gate 6 sign-off, not left only as a code comment.")
else:
    print("[OK] No demographic-adjacent keyword found in the real live 'Tags' values - Gate 1's "
          "'No Protected-Class Field in Scope' statement stands, re-verified live rather than "
          "assumed. 'Tags' remains excluded from the feature set regardless (low information).")

# --- Load only the feature + target columns, filtered to the real trainable classes
select_cols = FEATURE_COLS_CATEGORICAL + [COMPANY_COL, TARGET_COL]
df_pl = (
    gold_lazy.select(select_cols)
    .filter(pl.col(TARGET_COL).is_in(ORDINAL_CLASSES))
    .collect()
)
for barred in BARRED_COLUMNS:
    assert barred not in df_pl.columns, f"[CHECK FAILED] barred column '{barred}' present in the loaded feature frame."

row_count_drift = df_pl.height != documented_trainable_rows
if row_count_drift:
    print(f"[DRIFT DETECTED] Live trainable row count ({df_pl.height:,}) != Gate 2's documented "
          f"count ({documented_trainable_rows:,}).")
else:
    print(f"[OK] Live trainable row count ({df_pl.height:,}) matches Gate 2's documented count exactly.")

class_dist = (
    df_pl.group_by(TARGET_COL).agg(pl.len().alias("n")).sort("n", descending=True)
)
print(f"\n=== REAL FRICTION SEVERITY CLASS DISTRIBUTION (trainable rows only, n={df_pl.height:,}) ===")
print(class_dist)
minority_n = int(class_dist["n"].min())
majority_n = int(class_dist["n"].max())
print(f"[FINDING] Real class imbalance ratio (majority/minority): {majority_n / minority_n:.1f}:1 "
      "- macro-F1 is the only defensible champion-selection metric here, not accuracy.")

# --- Extract columns to plain Python/numpy (no Arrow conversion path) and build a pandas frame
feature_data = {col: df_pl[col].cast(pl.Utf8).fill_null("MISSING").to_list() for col in FEATURE_COLS_CATEGORICAL}
feature_data[COMPANY_COL] = df_pl[COMPANY_COL].cast(pl.Utf8).fill_null("MISSING").to_list()
target_data = df_pl[TARGET_COL].cast(pl.Utf8).to_list()
X_full = pd.DataFrame(feature_data)
y_full_labels = pd.Series(target_data, name=TARGET_COL)
print(f"\n[OK] Built feature frame: {X_full.shape[0]:,} rows x {X_full.shape[1]} columns "
      f"({FEATURE_COLS_CATEGORICAL + [COMPANY_COL]}).")

# ============================================================
# SECTION 6: Single stratified train/test split (CFPB has no provided split - Gate 1 policy)
# ============================================================
X_train_raw, X_test_raw, y_train_labels, y_test_labels = train_test_split(
    X_full, y_full_labels, test_size=0.20, stratify=y_full_labels, random_state=RANDOM_STATE
)
print(f"\n[OK] Stratified train/test split: train={len(X_train_raw):,}, test={len(X_test_raw):,} "
      f"(test_size=0.20, random_state={RANDOM_STATE}).")

label_encoder = LabelEncoder().fit(y_train_labels)
y_train = label_encoder.transform(y_train_labels)
y_test = label_encoder.transform(y_test_labels)
print(f"[OK] Target integer-encoded ({len(label_encoder.classes_)} classes: "
      f"{list(label_encoder.classes_)}) - same encoding used for every candidate.")

# ============================================================
# SECTION 7: Shared preprocessing for 5 of the 6 candidates - one-hot the 7 low-cardinality
# categorical columns (fit on TRAIN only), frequency-encode Company (fit on TRAIN only, unseen
# companies at test time get frequency 0 - never fabricated, never test-set-derived).
# ============================================================
ohe = ColumnTransformer(
    [("ohe", OneHotEncoder(handle_unknown="ignore", dtype=np.float32), FEATURE_COLS_CATEGORICAL)],
    remainder="drop",
)
X_train_ohe = ohe.fit_transform(X_train_raw)
X_test_ohe = ohe.transform(X_test_raw)

company_freq_map = X_train_raw[COMPANY_COL].value_counts().to_dict()
train_company_freq = X_train_raw[COMPANY_COL].map(company_freq_map).fillna(0).to_numpy(dtype=np.float32).reshape(-1, 1)
test_company_freq = X_test_raw[COMPANY_COL].map(company_freq_map).fillna(0).to_numpy(dtype=np.float32).reshape(-1, 1)

X_train_shared = sp.hstack([X_train_ohe, sp.csr_matrix(train_company_freq)], format="csr")
X_test_shared = sp.hstack([X_test_ohe, sp.csr_matrix(test_company_freq)], format="csr")
print(f"\n[OK] Shared preprocessed feature matrix: train={X_train_shared.shape}, test={X_test_shared.shape} "
      f"(one-hot on {len(FEATURE_COLS_CATEGORICAL)} columns + 1 Company frequency column, sparse).")


# ============================================================
# SECTION 8: Candidate model set (6 models, matching BP1's named set) - class_weight='balanced'
# applied wherever this environment's real installed library API supports it for multiclass.
# ============================================================
CANDIDATES = {
    "logistic_regression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=cv_settings["random_state"]
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=100, max_depth=20, class_weight="balanced", n_jobs=1,
        random_state=cv_settings["random_state"],
    ),
    # No class_weight equivalent in this environment's installed HistGradientBoostingClassifier -
    # documented library-capability asymmetry, not an oversight (see notebook markdown).
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        max_iter=100, random_state=cv_settings["random_state"]
    ),
    # No native multiclass class_weight in XGBoost's sklearn API - same documented asymmetry.
    "xgboost": XGBClassifier(
        n_estimators=100, max_depth=6, n_jobs=1, verbosity=0, random_state=cv_settings["random_state"]
    ),
    "lightgbm": LGBMClassifier(
        n_estimators=100, class_weight="balanced", n_jobs=1, verbose=-1,
        random_state=cv_settings["random_state"],
    ),
}

# Real environment quirk already discovered on this exact installed sklearn version (BP1 Gate 3):
# HistGradientBoostingClassifier rejects sparse input. float32 (not float64) keeps the transient
# densified matrix within the WARP RAM ceiling (WARP: float32 over defaults for reused arrays).
NEEDS_DENSE = {"hist_gradient_boosting"}

# ============================================================
# SECTION 8b: Comprehensive per-candidate concurrency hardening - added 2026-09-22 after a real
# Windows crash (Kernel-Power Event 41, unclean shutdown) while this notebook was running for the
# first time. Full incident writeup: LESSONS_LEARNED_APPLIED.md Lesson #21. This supersedes an
# earlier, narrower fix that capped ONLY hist_gradient_boosting's CV step - the broader review
# below covers every candidate, not just the one already identified, per explicit instruction.
#
# Why every candidate is in scope, not just hist_gradient_boosting: every model in CANDIDATES
# already has its OWN internal parallelism pinned to 1 (n_jobs=1 / thread_count=1, Section 8), so
# the only concurrency left anywhere in this notebook is at the CV-fold level, set by the
# `n_jobs` passed to `cross_validate` below under the threading backend. Before this fix that
# value was the full N_JOBS (=16, a SYNTHETIC hardware-benchmark recommendation, not a number
# derived from this workload) for every candidate except hist_gradient_boosting - meaning up to
# min(N_JOBS, cv_settings['n_splits'])=5 folds could already run fully concurrently for the other
# five candidates too. That is real, uncapped concurrency this review had not previously flagged.
#
# Two risk categories, addressed separately per the user's own framing:
#   (a) PEAK MEMORY - a model whose per-fold data footprint is large multiplied by concurrent
#       folds. hist_gradient_boosting is the only candidate with a known large per-fold footprint
#       (its forced dense float32 copy, ~2.4GB by code-review estimate, never profiled for real).
#   (b) SUSTAINED CPU / THERMAL LOAD - independent of any single candidate's memory footprint:
#       this notebook's real dataset (816,717 trainable rows) is roughly 63x larger than any
#       prior real run in this project (BP1 Gate 3 benchmarked ~13,083 rows), so total wall-clock
#       time at high concurrent CPU utilization across all 6 candidates is far longer than
#       anything previously run for real here - a real risk this project has no prior precedent
#       for, called out explicitly by the user as distinct from RAM/CPU peaks. psutil does not
#       expose a portable CPU temperature reading on Windows, so there is no live thermal signal
#       to check against; the mitigation below (a fixed inter-candidate pause) is a precaution,
#       not a measured response, and is documented as such rather than presented as verified.
#
# Model hyperparameters (n_estimators, max_depth, iterations, etc.) are deliberately left
# untouched by this fix - reducing them would compromise the real benchmark this notebook exists
# to run. Every change below is concurrency/pacing/monitoring only.
CV_N_JOBS_OVERRIDE = {
    # Already the candidate running when the real crash occurred. Tightened from the previously
    # applied n_jobs=2 down to n_jobs=1 (no CV-fold concurrency at all for this one candidate)
    # rather than trust a partial mitigation a second time on a confirmed real incident.
    "hist_gradient_boosting": 1,
    # Sparse input (cheap per-fold subsetting), but 100 trees x max_depth=20 is the heaviest
    # per-fold CPU workload among the sparse-input candidates - capped rather than left uncapped.
    "random_forest": 2,
}
# Conservative default for every candidate not explicitly listed above (logistic_regression,
# xgboost, lightgbm): sparse input, cheap per-fold subsetting, low individual risk - but still
# capped well below the uncapped N_JOBS, since cv_settings['n_splits']==5 makes any value above 5
# pointless anyway, and this dataset's real scale has no prior real-run precedent in this project.
DEFAULT_CV_N_JOBS = min(4, N_JOBS)
# Fixed precautionary pause between candidates - addresses sustained-load/thermal risk, which is
# a distinct category from peak RAM and is not covered by assert_within_ram_ceiling() at all.
COOLDOWN_SECONDS_BETWEEN_CANDIDATES = 20
# If live headroom is already below this when a candidate is ABOUT TO START, force that
# candidate's CV to n_jobs=1 regardless of its planned cap above - never assumes a prior
# candidate's cleanup (Section 9's `del X_cv` + assert_within_ram_ceiling) fully freed memory.
MIN_HEADROOM_GB_BEFORE_CANDIDATE = 4.0

# ============================================================
# SECTION 9: Identical CV folds across every candidate (Gate 3 exit criterion)
# ============================================================
skf = StratifiedKFold(
    n_splits=cv_settings["n_splits"], shuffle=cv_settings["shuffle"], random_state=cv_settings["random_state"]
)
SCORING = ["f1_macro", "f1_weighted", "accuracy"]

cv_results_rows = []
failed_candidates = []
candidate_names = list(CANDIDATES.keys())
for candidate_idx, (name, model) in enumerate(CANDIDATES.items()):
    X_cv, y_cv = X_train_shared, y_train
    if name in NEEDS_DENSE:
        X_cv = np.asarray(X_cv.todense(), dtype=np.float32)

    # Adaptive pre-candidate check (Section 8b): never assume a prior candidate's cleanup left
    # enough headroom - read it live, right before this candidate's CV starts, not just after
    # the previous one finished.
    planned_cv_n_jobs = CV_N_JOBS_OVERRIDE.get(name, DEFAULT_CV_N_JOBS)
    headroom_before_gb = memory_headroom_gb(RESOURCE_LIMITS["ceilings"]["max_ram_fraction"])
    if headroom_before_gb < MIN_HEADROOM_GB_BEFORE_CANDIDATE:
        cv_n_jobs = 1
        print(f"[WARP] {name}: live headroom {headroom_before_gb}GB is below the "
              f"{MIN_HEADROOM_GB_BEFORE_CANDIDATE}GB pre-candidate threshold - forcing n_jobs=1 "
              f"(planned was {planned_cv_n_jobs}).")
    else:
        cv_n_jobs = planned_cv_n_jobs

    print(f"\n[BENCH] starting {name} ({candidate_idx + 1}/{len(candidate_names)}, "
          f"{cv_settings['n_splits']}-fold CV, n_jobs={cv_n_jobs}, threading backend, "
          f"headroom={headroom_before_gb}GB)...")
    t0 = time.perf_counter()
    try:
        with joblib.parallel_backend("threading", n_jobs=cv_n_jobs):
            scores = cross_validate(model, X_cv, y_cv, cv=skf, scoring=SCORING, n_jobs=cv_n_jobs)
        elapsed = time.perf_counter() - t0
        row = {
            "model": name,
            "status": "OK",
            "elapsed_seconds": round(elapsed, 2),
            "cv_n_jobs_used": cv_n_jobs,
            "ram_headroom_gb_before_candidate": headroom_before_gb,
            "mean_f1_macro": round(float(np.mean(scores["test_f1_macro"])), 4),
            "std_f1_macro": round(float(np.std(scores["test_f1_macro"])), 4),
            "mean_f1_weighted": round(float(np.mean(scores["test_f1_weighted"])), 4),
            "mean_accuracy": round(float(np.mean(scores["test_accuracy"])), 4),
        }
        print(f"[BENCH] {name} done in {elapsed:.1f}s: mean f1_macro={row['mean_f1_macro']} "
              f"(+/- {row['std_f1_macro']})")
    except Exception as e:  # noqa: BLE001 - continue gracefully on an individual model failure
        elapsed = time.perf_counter() - t0
        row = {
            "model": name, "status": f"FAILED: {type(e).__name__}: {e}", "elapsed_seconds": round(elapsed, 2),
            "cv_n_jobs_used": cv_n_jobs, "ram_headroom_gb_before_candidate": headroom_before_gb,
            "mean_f1_macro": None, "std_f1_macro": None, "mean_f1_weighted": None, "mean_accuracy": None,
        }
        failed_candidates.append(name)
        print(f"[FAILED] {name} after {elapsed:.1f}s: {type(e).__name__}: {e} - continuing with remaining models.")
    cv_results_rows.append(row)
    del X_cv  # free the (possibly dense, ~2.4GB) per-candidate matrix before the next candidate starts
    assert_within_ram_ceiling(RESOURCE_LIMITS)

    if candidate_idx < len(candidate_names) - 1:
        # Precautionary pause (Section 8b) - sustained-load/thermal mitigation, not a
        # RAM-ceiling check (assert_within_ram_ceiling above already covers that separately).
        print(f"[WARP] cooling down {COOLDOWN_SECONDS_BETWEEN_CANDIDATES}s before the next "
              f"candidate (precautionary pacing, not a measured thermal reading)...")
        time.sleep(COOLDOWN_SECONDS_BETWEEN_CANDIDATES)

cv_results_df = pd.DataFrame(cv_results_rows)
print("\n=== CV BENCHMARK RESULTS (identical folds across all candidates) ===")
print(cv_results_df.to_string(index=False))

results_csv_path = ARTIFACTS_DIR / "gate3_cv_benchmark_results.csv"
cv_results_df.to_csv(results_csv_path, index=False)
print(f"\n[SAVED] {results_csv_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 10: Champion selection - highest mean CV f1_macro (severe class imbalance -> macro F1)
# ============================================================
passing_results = cv_results_df[cv_results_df["status"] == "OK"]
assert len(passing_results) > 0, "[CHECK FAILED] Every candidate model failed - nothing to select a champion from."
champion_row = passing_results.loc[passing_results["mean_f1_macro"].idxmax()]
champion_name = champion_row["model"]
print(f"\n[RESULT] Champion: {champion_name} (mean CV f1_macro={champion_row['mean_f1_macro']})")

# ============================================================
# SECTION 11: Refit champion on the FULL train split, evaluate ONCE on the held-out test split
# ============================================================
assert_within_ram_ceiling(RESOURCE_LIMITS)
X_train_final, X_test_final = X_train_shared, X_test_shared
if champion_name in NEEDS_DENSE:
    X_train_final = np.asarray(X_train_final.todense(), dtype=np.float32)
    X_test_final = np.asarray(X_test_final.todense(), dtype=np.float32)
y_train_final, y_test_final_labels = y_train, y_test_labels.to_numpy()

champion_model = CANDIDATES[champion_name]
print(f"\n[FINAL] Refitting champion ({champion_name}) on the full train split ({len(X_train_raw):,} rows)...")
t0 = time.perf_counter()
champion_model.fit(X_train_final, y_train_final)
fit_elapsed = time.perf_counter() - t0
print(f"[FINAL] Fit done in {fit_elapsed:.1f}s. Evaluating ONCE on the held-out test split "
      f"({len(X_test_raw):,} rows)...")

y_pred_raw = champion_model.predict(X_test_final)
y_pred = label_encoder.inverse_transform(np.asarray(y_pred_raw).astype(int).ravel())
y_test_labels_arr = y_test_final_labels

test_report = classification_report(y_test_labels_arr, y_pred, output_dict=True, zero_division=0)
test_f1_macro = test_report["macro avg"]["f1-score"]
test_f1_weighted = test_report["weighted avg"]["f1-score"]
test_accuracy = test_report["accuracy"]
print(f"[FINAL] Held-out test: f1_macro={test_f1_macro:.4f}, f1_weighted={test_f1_weighted:.4f}, "
      f"accuracy={test_accuracy:.4f}")

labels_sorted = sorted(set(y_test_labels_arr) | set(y_pred))
cm = confusion_matrix(y_test_labels_arr, y_pred, labels=labels_sorted)
cm_df = pd.DataFrame(cm, index=labels_sorted, columns=labels_sorted)

confused_pairs = []
for i, true_label in enumerate(labels_sorted):
    for j, pred_label in enumerate(labels_sorted):
        if i != j and cm[i, j] > 0:
            confused_pairs.append((true_label, pred_label, int(cm[i, j])))
confused_pairs.sort(key=lambda t: t[2], reverse=True)
print("\n[FINAL] Confused (true -> predicted) pairs on the held-out test split (4-class, all shown):")
for true_label, pred_label, count in confused_pairs:
    print(f"  {true_label} -> {pred_label}: {count}")

# ============================================================
# SECTION 12: Write outputs (idempotent overwrite-in-place)
# ============================================================
report_json_path = ARTIFACTS_DIR / "gate3_champion_test_classification_report.json"
with open(report_json_path, "w", encoding="utf-8") as f:
    json.dump(test_report, f, indent=2)
print(f"\n[SAVED] {report_json_path.relative_to(PROJECT_ROOT)}")

cm_csv_path = ARTIFACTS_DIR / "gate3_champion_test_confusion_matrix.csv"
cm_df.to_csv(cm_csv_path)
print(f"[SAVED] {cm_csv_path.relative_to(PROJECT_ROOT)}")

model_inventory_entry = {
    "bp_id": "bp2",
    "gate": 3,
    "compliance_touchpoint": "Model inventory entry opened (SR 11-7 first-line record)",
    "model_name": champion_name,
    "model_family": "one-hot/frequency-encoded structured features + " + champion_name,
    "target_variable": TARGET_COL,
    "feature_columns": FEATURE_COLS_CATEGORICAL + [COMPANY_COL],
    "barred_columns": BARRED_COLUMNS,
    "excluded_classes_not_trained_on": EXCLUDED_CLASSES,
    "training_data": "Real CFPB extract, fresh stratified 80/20 split (CFPB provides no split of "
                      "its own), random_state=" + str(RANDOM_STATE),
    "n_train_rows": len(X_train_raw),
    "n_test_rows": len(X_test_raw),
    "n_classes": len(label_encoder.classes_),
    "class_imbalance_ratio_majority_over_minority": round(majority_n / minority_n, 1),
    "cv_folds": cv_settings["n_splits"],
    "cv_mean_f1_macro": float(champion_row["mean_f1_macro"]),
    "held_out_test_f1_macro": float(test_f1_macro),
    "held_out_test_f1_weighted": float(test_f1_weighted),
    "held_out_test_accuracy": float(test_accuracy),
    "candidates_evaluated": list(CANDIDATES.keys()),
    "candidates_failed": failed_candidates,
    "demographic_adjacent_tags_found": demographic_adjacent_tags_found,
    "open_item_company_public_response_ablation_not_run": True,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "first-line record only - Gate 4 independent-style statistical validation not yet performed",
}
inventory_path = ARTIFACTS_DIR / "model_inventory_entry.json"
with open(inventory_path, "w", encoding="utf-8") as f:
    json.dump(model_inventory_entry, f, indent=2)
print(f"[SAVED] {inventory_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 13: Write the Gate 3 config block (marker-based, order-independent, reuses BP1's
# bp1_config_sync.py unmodified)
# ============================================================
import re  # noqa: E402
from utils.bp1_config_sync import write_gate_block  # noqa: E402

status_text = BP2_CONFIG_PATH.read_text(encoding="utf-8")
current_status_line = [ln for ln in status_text.splitlines() if ln.startswith("status:")][0]
new_status_value = current_status_line.split('"')[1] + "_gate3_confirmed" \
    if "_gate3_confirmed" not in current_status_line else current_status_line.split('"')[1]
status_text = re.sub(r'^status:.*$', f'status: "{new_status_value}"', status_text, count=1, flags=re.MULTILINE)
BP2_CONFIG_PATH.write_text(status_text, encoding="utf-8")

gate3_marker = "# --- Gate 3 (Model/Classifier Benchmark) results (appended, idempotent overwrite) ---"
gate3_block_lines = [
    "gate3_model_benchmark:",
    f'  champion_model: "{champion_name}"',
    f"  cv_mean_f1_macro: {champion_row['mean_f1_macro']}",
    f"  held_out_test_f1_macro: {round(test_f1_macro, 4)}",
    f"  held_out_test_f1_weighted: {round(test_f1_weighted, 4)}",
    f"  held_out_test_accuracy: {round(test_accuracy, 4)}",
    f"  class_imbalance_ratio: {round(majority_n / minority_n, 1)}",
    f"  candidates_evaluated: {list(CANDIDATES.keys())}",
    f"  candidates_failed: {failed_candidates}",
    f"  demographic_adjacent_tags_found: {demographic_adjacent_tags_found}",
    "  open_item_company_public_response_ablation_not_run: true",
    f'  generated_at_utc: "{datetime.now(timezone.utc).isoformat()}"',
]
write_gate_block(BP2_CONFIG_PATH, gate3_marker, gate3_block_lines)
print(f"[SAVED] {BP2_CONFIG_PATH.relative_to(PROJECT_ROOT)} (gate3_model_benchmark block)")

# ============================================================
# SECTION 14: Structural integrity checks - raise AssertionError, never silently pass
# ============================================================
checks = {
    "at_least_one_candidate_passed": len(passing_results) > 0,
    "identical_cv_object_used_for_all_candidates": True,  # by construction - same `skf` instance, Section 9
    "champion_selected_by_mean_cv_f1_macro": champion_name in CANDIDATES,
    "held_out_test_touched_exactly_once": True,  # by construction - Section 11 is the only test-set use
    "no_barred_column_in_feature_frame": all(b not in df_pl.columns for b in BARRED_COLUMNS),
    "cv_results_csv_written": results_csv_path.exists(),
    "classification_report_json_written": report_json_path.exists(),
    "confusion_matrix_csv_written": cm_csv_path.exists(),
    "model_inventory_entry_written": inventory_path.exists(),
    "bp2_config_yaml_updated": BP2_CONFIG_PATH.exists(),
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print(f"\n[ALL CHECKS PASSED] BP2 Gate 3 complete. Champion: {champion_name} "
      f"(held-out test f1_macro={round(test_f1_macro, 4)}, class imbalance ratio "
      f"{round(majority_n / minority_n, 1)}:1). {len(failed_candidates)} candidate(s) failed: "
      f"{failed_candidates or 'none'}. Company public response ablation NOT run (open item). "
      "Proceed to BP2 Gate 4 (Statistical Validation & Explainability) next.")
